This notebook is based on TCCON_analysis in emissions_data folder

In [1]:
from netCDF4 import Dataset as NetCDFFile
import netCDF4 as ncd
import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import cftime as cft
import time
import json
import glob

import cartopy.crs as ccrs

In [2]:
from __future__ import print_function
import argparse

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.autograd import Variable
from torch.utils.data import DataLoader
from model import Net as DBPNLL
from data import get_eval_set
from functools import reduce
from skimage.transform import pyramid_reduce, pyramid_expand

import scipy.io as sio

In [3]:
def normalize_array(data):
    vmax = np.amax(data[np.nonzero(data)])
    vmin = np.amin(data[np.nonzero(data)])
    range_data = vmax - vmin
    
    normalized_data = (data - vmin)/range_data
    threshold = - 1./range_data
    data = np.maximum(normalized_data,threshold)
    data[data == threshold] = -1
    
    return data

In [4]:
def find_nearest(array, value):
    array = np.asarray(array)
    idx = (np.abs(array - value)).argmin()
    return idx

In [5]:
# The boundary format is low_lat, left_lon, high_lat and right_lon
def arrays_centered_on_factory(coordinates, day, lat, lon, size, xco2_array):
    nb_polluters = len(coordinates)
    for indx in range(nb_polluters):
        boundaries = np.zeros(4)
        boundaries[0] = find_nearest(lat, float(coordinates[indx, 1]) - size)
        boundaries[1] = find_nearest(lon, float(coordinates[indx, 2]) - size)
        boundaries[2] = find_nearest(lat, float(coordinates[indx, 1]) + size)
        boundaries[3] = find_nearest(lon, float(coordinates[indx, 2]) + size)
        boundaries = boundaries.astype(int)
        np.save('/data/andria/Input/emitters/centered_{}_{}.npy'.format(day, indx), xco2_array[boundaries[0]:boundaries[2],
                                                                                           boundaries[1]:boundaries[3]])

In [6]:
def save_img(img, img_name):
    save_arr = img.numpy().squeeze(axis = 0).squeeze(axis = 0)
    # save img
    save_dir=os.path.join('/data/andria/Results/','emitters')
    if not os.path.exists(save_dir):
        os.makedirs(save_dir)
        
    save_fn = save_dir +'/'+ img_name
    np.save(save_fn, save_arr)

In [7]:
coordinates = np.load('../../../emissions_data/coordinates_emitters.npy', allow_pickle = True)

In [8]:
#4 days per year to account for seasonality
list_of_days = ['20150120', '20150420', '20150720', '20151020',
               '20160120', '20160420', '20160720', '20161020',
               '20170120', '20170420', '20170720', '20171020',
               '20180120', '20180420', '20180720', '20181020',
               '20190120', '20190420', '20190720', '20191020',
               '20200120', '20200420', '20200720', '20201020',
               '20210120', '20210420', '20210720', '20211020']

In [9]:
model_w_16 = 'weights/MOD_tensorese-hivemindDBPNLL1channel_16_MSE.pth'

gpus_list=range(1)
cuda = True
if cuda and not torch.cuda.is_available():
    raise Exception("No GPU found, please run without --cuda")

torch.manual_seed(123)
if cuda:
    torch.cuda.manual_seed(123)
    
print('===> Building models')
model_16 = DBPNLL(num_channels=1, base_filter=64,  feat = 256, num_stages=10, scale_factor=16)

if cuda:
    model_16 = torch.nn.DataParallel(model_16, device_ids=gpus_list)

model_16.load_state_dict(torch.load(model_w_16, map_location=lambda storage, loc: storage))
print('Pre-trained SR model is loaded.')

if cuda:
    model_16 = model_16.cuda(gpus_list[0])

===> Building models
Pre-trained SR model is loaded.


In [11]:
for day in list_of_days:
    filename = 'oco2_GEOS_L3CO2_day_'+day+'_B10206Ar.nc4'
    print("===> Processing: %s " % (filename))
    file = NetCDFFile('/data/andria/OCO-2/'+filename)
    to_save = filename.replace('oco2_GEOS_L3CO2_day_', '').replace('_B10206Ar.nc4', '')
    xco2 = np.asarray(file.variables['XCO2'][:])
    xco2 = np.squeeze(xco2)
    lat = np.asarray(file.variables['lat'][:])
    lon = np.asarray(file.variables['lon'][:])

    xco2_norm = normalize_array(xco2)

    arrays_centered_on_factory(coordinates, to_save, lat, lon, 10, xco2_norm)

    print('===> Loading datasets')
    test_set = get_eval_set(os.path.join('/data/andria/Input','emitters'), 16)
    testing_data_loader = DataLoader(dataset=test_set, num_workers=1, batch_size=1, shuffle=False)

    def eval():
        model_16.eval()
        for batch in testing_data_loader:
            with torch.no_grad():
                input, name = Variable(batch[0]), batch[1]
                print(input.size())
                return ''
            if cuda:
                input = input.cuda(gpus_list[0])

            t0 = time.time()
            with torch.no_grad():
                prediction_16 = model_16(input)
            t1 = time.time()
            name_16 = name[0].replace('.npy', '_16.npy')
            print("===> Processing: %s || Path: 1-16 || Timer: %.4f sec." % (name_16, (t1 - t0)))

            save_img(prediction_16.cpu().data, name_16)

    eval()
    files = glob.glob('/data/andria/Input/emitters/*')
    for f in files:
        os.remove(f)

===> Processing: oco2_GEOS_L3CO2_day_20150120_B10206Ar.nc4 
===> Loading datasets
torch.Size([1, 1, 40, 32])
===> Processing: oco2_GEOS_L3CO2_day_20150420_B10206Ar.nc4 
===> Loading datasets
torch.Size([1, 1, 40, 32])
===> Processing: oco2_GEOS_L3CO2_day_20150720_B10206Ar.nc4 
===> Loading datasets
torch.Size([1, 1, 40, 32])
===> Processing: oco2_GEOS_L3CO2_day_20151020_B10206Ar.nc4 
===> Loading datasets
torch.Size([1, 1, 40, 32])
===> Processing: oco2_GEOS_L3CO2_day_20160120_B10206Ar.nc4 
===> Loading datasets
torch.Size([1, 1, 40, 32])
===> Processing: oco2_GEOS_L3CO2_day_20160420_B10206Ar.nc4 
===> Loading datasets
torch.Size([1, 1, 40, 32])
===> Processing: oco2_GEOS_L3CO2_day_20160720_B10206Ar.nc4 
===> Loading datasets
torch.Size([1, 1, 40, 32])
===> Processing: oco2_GEOS_L3CO2_day_20161020_B10206Ar.nc4 
===> Loading datasets
torch.Size([1, 1, 40, 32])
===> Processing: oco2_GEOS_L3CO2_day_20170120_B10206Ar.nc4 
===> Loading datasets
torch.Size([1, 1, 40, 32])
===> Processing: oc